<a href="https://colab.research.google.com/github/irullah/nlp-abstract-analysis/blob/main/text_mining_klastering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install PySastrawi

import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score

nltk.download('stopwords')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.2/211.2 kB 4.2 MB/s eta 0:00:00


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [2]:
# Sesuaikan path jika dataset ada di dalam folder 'data'
df_raw = pd.read_csv('dataset-abstrak.csv')
df = df_raw[['Kategori', 'Abstrak']].copy()
df.head()

,Kategori,Abstrak
0,RPL,Sistem informasi akademik (SIAKAD) merupaka...
1,RPL,Berjalannya koneksi jaringan komputer dengan l...
2,RPL,Web server adalah sebuah perangkat lunak serve...
3,KOMPUTASI,Penjadwalan kuliah di Perguruan Tinggi me...
4,RPL,Seiring perkembangan teknologi yang ada diduni...


In [3]:
def clean_text(text):
    text = str(text).lower() # Case folding
    text = re.sub(r"http\S+", "", text) # Hapus URL
    text = re.sub(r"\s—\s", " ", text) # Hapus dash
    # Hapus angka dan tanda baca menggunakan regex (lebih optimal dari looping)
    text = re.sub(r"[0-9,!\"#$%&()*+-.…/:;<=>?@[\]^_`{|}~\n]", " ", text)
    # Hapus spasi berlebih
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['abstrak_clean'] = df['Abstrak'].apply(clean_text)
df[['Abstrak', 'abstrak_clean']].head()

,Abstrak,abstrak_clean
0,Sistem informasi akademik (SIAKAD) merupaka...,sistem informasi akademik siakad merupakan sis...
1,Berjalannya koneksi jaringan komputer dengan l...,berjalannya koneksi jaringan komputer dengan l...
2,Web server adalah sebuah perangkat lunak serve...,web server adalah sebuah perangkat lunak serve...
3,Penjadwalan kuliah di Perguruan Tinggi me...,penjadwalan kuliah di perguruan tinggi merupak...
4,Seiring perkembangan teknologi yang ada diduni...,seiring perkembangan teknologi yang ada diduni...


In [4]:
stop_words = set(stopwords.words('indonesian'))

def remove_stopwords(text):
    # Split teks sekaligus memisahkannya ke dalam token (list)
    tokens = text.split()
    filtered_tokens = [word for word in tokens if word not in stop_words]
    return " ".join(filtered_tokens)

df['abstrak_no_stopword'] = df['abstrak_clean'].apply(remove_stopwords)
df[['abstrak_clean', 'abstrak_no_stopword']].head()

,abstrak_clean,abstrak_no_stopword
0,sistem informasi akademik siakad merupakan sis...,sistem informasi akademik siakad sistem inform...
1,berjalannya koneksi jaringan komputer dengan l...,berjalannya koneksi jaringan komputer lancar g...
2,web server adalah sebuah perangkat lunak serve...,web server perangkat lunak server berfungsi me...
3,penjadwalan kuliah di perguruan tinggi merupak...,penjadwalan kuliah perguruan kompleks permasal...
4,seiring perkembangan teknologi yang ada diduni...,seiring perkembangan teknologi didunia muncul ...


In [5]:
factory = StemmerFactory()
stemmer = factory.create_stemmer()

# Menggunakan dictionary cache agar stemming tidak memproses kata yang sama berulang kali
term_dict = {}

def stem_text(text):
    tokens = text.split()
    stemmed_tokens = []
    for term in tokens:
        if term not in term_dict:
            term_dict[term] = stemmer.stem(term)
        stemmed_tokens.append(term_dict[term])
    return " ".join(stemmed_tokens)

df['abstrak_stem'] = df['abstrak_no_stopword'].apply(stem_text)
df[['abstrak_no_stopword', 'abstrak_stem']].head()

,abstrak_no_stopword,abstrak_stem
0,sistem informasi akademik siakad sistem inform...,sistem informasi akademik siakad sistem inform...
1,berjalannya koneksi jaringan komputer lancar g...,jalan koneksi jaring komputer lancar ganggu ha...
2,web server perangkat lunak server berfungsi me...,web server perangkat lunak server fungsi terim...
3,penjadwalan kuliah perguruan kompleks permasal...,jadwal kuliah guru kompleks masalah variabel t...
4,seiring perkembangan teknologi didunia muncul ...,iring kembang teknologi dunia muncul teknologi...


In [6]:
X_text = df['abstrak_stem']
y_true = df['Kategori'].values

tfidf_vectorizer = TfidfVectorizer()
X_tfidf = tfidf_vectorizer.fit_transform(X_text)

# Konversi ke DataFrame untuk melihat fitur
df_tfidf = pd.DataFrame(X_tfidf.toarray(), columns=tfidf_vectorizer.get_feature_names_out(), index=df.index)
df_tfidf.head()

,aalysis,aam,abad,abadi,abai,abdi,ability,abjad,absah,absensi,...,zara,zat,zcz,zf,zona,zone,zoning,zoom,zucara,zungu
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [7]:
# Menambahkan random_state agar hasil PCA konsisten setiap kali dijalankan
pca = PCA(n_components=10, random_state=42)
X_pca = pca.fit_transform(X_tfidf.toarray())

print("Total Variance Explained:", sum(pca.explained_variance_ratio_))

Total Variance Explained: 0.11356195450091265


In [8]:
kmeans = KMeans(n_clusters=2, random_state=42)
kmeans.fit(X_pca)

df['Cluster_Label'] = kmeans.labels_
print("Center Clusters:\n", kmeans.cluster_centers_)

Center Clusters:
 [[ 1.11653908e-02 -2.18012661e-02 -3.74133789e-03  5.61191827e-03
   2.63873248e-04 -8.58902297e-03 -6.78676240e-04 -3.34759757e-04
  -9.78227979e-04 -2.57347759e-03]
 [-1.97861111e-01  3.86338716e-01  6.62999877e-02 -9.94484121e-02
  -4.67607942e-03  1.52205477e-01  1.20267743e-02  5.93225430e-03
   1.73351098e-02  4.56044168e-02]]


In [10]:
def map_cluster(label):
    return "KOMPUTASI" if label == 0 else "RPL"

df['Prediksi'] = df['Cluster_Label'].apply(map_cluster)

# Cetak perbandingan prediksi vs aktual
for index, row in df.iterrows():
    print(f"Prediksi: {row['Prediksi']} \t| Aktual: {row['Kategori']}")

print("\nAccuracy Score setelah label disesuaikan:", accuracy_score(y_true, df['Prediksi']))

Prediksi: KOMPUTASI 	| Aktual: RPL
Prediksi: KOMPUTASI 	| Aktual: RPL
Prediksi: KOMPUTASI 	| Aktual: RPL
Prediksi: KOMPUTASI 	| Aktual: KOMPUTASI
Prediksi: KOMPUTASI 	| Aktual: RPL
Prediksi: KOMPUTASI 	| Aktual: KOMPUTASI
Prediksi: RPL 	| Aktual: KOMPUTASI
Prediksi: KOMPUTASI 	| Aktual: KOMPUTASI
Prediksi: RPL 	| Aktual: KOMPUTASI
Prediksi: KOMPUTASI 	| Aktual: RPL
Prediksi: KOMPUTASI 	| Aktual: RPL
Prediksi: KOMPUTASI 	| Aktual: KOMPUTASI
Prediksi: KOMPUTASI 	| Aktual: KOMPUTASI
Prediksi: KOMPUTASI 	| Aktual: KOMPUTASI
Prediksi: KOMPUTASI 	| Aktual: KOMPUTASI
Prediksi: KOMPUTASI 	| Aktual: KOMPUTASI
Prediksi: KOMPUTASI 	| Aktual: KOMPUTASI
Prediksi: KOMPUTASI 	| Aktual: KOMPUTASI
Prediksi: KOMPUTASI 	| Aktual: KOMPUTASI
Prediksi: KOMPUTASI 	| Aktual: KOMPUTASI
Prediksi: KOMPUTASI 	| Aktual: KOMPUTASI
Prediksi: KOMPUTASI 	| Aktual: KOMPUTASI
Prediksi: KOMPUTASI 	| Aktual: KOMPUTASI
Prediksi: KOMPUTASI 	| Aktual: KOMPUTASI
Prediksi: KOMPUTASI 	| Aktual: KOMPUTASI
Prediksi: KOMPUTASI 	| 